# Paralelna cjevovodna mreža: predvidi → izračunaj → provjeri

**Poglavlje 13 — Gubitci, cjevovodi, crpke i mreže**

Tri grane spajaju ista dva čvora. Nepoznati su zajednički gubitak energije \(H\) i tri protoka. Faktor trenja ovisi o Reynoldsovu broju, pa sustav nije samo linearna podjela ukupnog protoka.

## 1. Predvidi

1. Koja će grana preuzeti najveći protok: najkraća, najšira ili najglađa?
2. Moraju li protoci biti jednaki zato što je pad energije jednak?
3. Koja dva neovisna reziduala treba pratiti: čvorni i gransko-energetski?

Model koristi Darcy–Weisbachovu jednadžbu, lokalne gubitke s koeficijentima \(\xi\) i eksplicitnu Swamee–Jainovu turbulentnu aproksimaciju koeficijenta trenja. Ovaj pokus rješava samo mreže u kojima su sve grane turbulentne (Re > 4000); prijelazno područje ne interpolira. Sintetički nastavni ulazi nisu mjerenja niti zamjena za kalibraciju stvarnog sustava. Nakon pokusa slijede zasebni podatci Z3 i Z6 iz poglavlja 13.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
g, nu = 9.81, 1.0e-6

branches = [
    {"name": "A", "L": 95.0, "D": 0.090, "eps": 0.00005, "xi": 2.0},
    {"name": "B", "L": 70.0, "D": 0.075, "eps": 0.00015, "xi": 3.5},
    {"name": "C", "L": 130.0, "D": 0.105, "eps": 0.00010, "xi": 1.5},
]

def friction_factor(Re, rel_roughness):
    if Re <= 4000 or rel_roughness < 0:
        raise ValueError("Ovaj mrežni pokus traži Re > 4000 i k_s/D ≥ 0; prijelaz se ne interpolira.")
    return 0.25/np.log10(rel_roughness/3.7 + 5.74/Re**0.9)**2

def minimum_turbulent_flow(branch):
    return 4001*nu*np.pi*branch["D"]/4

def branch_loss(Q, branch):
    if Q <= 0:
        return 0.0
    area = np.pi*branch["D"]**2/4
    velocity = Q/area
    Re = velocity*branch["D"]/nu
    lam = friction_factor(Re, branch["eps"]/branch["D"])
    return (lam*branch["L"]/branch["D"] + branch["xi"])*velocity**2/(2*g)

def flow_for_head(head, branch, q_upper, iterations=70):
    lo, hi = minimum_turbulent_flow(branch), q_upper
    if head < branch_loss(lo, branch)-1e-12:
        raise ValueError("Zadana visina nema obuhvaćeno turbulentno rješenje.")
    while branch_loss(hi, branch) < head:
        hi *= 2
    for _ in range(iterations):
        mid = 0.5*(lo+hi)
        if branch_loss(mid, branch) < head:
            lo = mid
        else:
            hi = mid
    return 0.5*(lo+hi)

def solve_network(Q_total, branch_data, tolerance=1e-11, max_iterations=100):
    low = max(branch_loss(minimum_turbulent_flow(b), b) for b in branch_data)
    if sum(flow_for_head(low, b, Q_total) for b in branch_data) > Q_total:
        raise ValueError("Traženi protok nije obuhvaćen modelom sa svim turbulentnim granama.")
    high = max(branch_loss(Q_total, b) for b in branch_data)
    history = []
    for iteration in range(max_iterations):
        head = 0.5*(low+high)
        flows = np.array([flow_for_head(head, b, Q_total) for b in branch_data])
        residual = flows.sum()-Q_total
        history.append((iteration, head, residual))
        if abs(residual) < tolerance:
            return head, flows, np.asarray(history)
        if residual > 0:
            high = head
        else:
            low = head
    raise RuntimeError("Mrežni rješavač nije konvergirao.")

Q_total = 0.030
head, flows, history = solve_network(Q_total, branches)
losses = np.array([branch_loss(q, b) for q, b in zip(flows, branches)])
for b, q, h in zip(branches, flows, losses):
    print(f"Grana {b['name']}: Q={1e3*q:6.3f} L/s, h={h:.6f} m")
print(f"Čvorni rezidual = {flows.sum()-Q_total:.3e} m³/s; zajednički H = {head:.6f} m")

## 2. Izračunaj: nelinearno rješenje i osjetljivost

Vanjska bisekcija mijenja zajednički \(H\) dok zbroj granskih protoka ne zatvori čvor. Unutarnja bisekcija za svaku granu invertira nelinearnu funkciju \(h_i(Q_i)\). Povijest vanjskog reziduala pokazuje stvarnu konvergenciju, a ne samo konačan broj.

Zatim mijenjamo promjer grane B. Za svaki novi promjer ponovno rješavamo cijelu mrežu; ostali protoci se također moraju prilagoditi.


In [ ]:
D2_values = np.linspace(0.065, 0.095, 31)
flow_sensitivity = []
for diameter in D2_values:
    modified = [dict(b) for b in branches]
    modified[1]["D"] = diameter
    _, q_modified, _ = solve_network(Q_total, modified)
    flow_sensitivity.append(q_modified)
flow_sensitivity = np.asarray(flow_sensitivity)

print(f"Promjena Q_B: {1e3*flow_sensitivity[0,1]:.3f} → {1e3*flow_sensitivity[-1,1]:.3f} L/s")
print(f"Broj vanjskih iteracija osnovnog slučaja: {len(history)}")


## 3. Provjeri

Čvorna bilanca provjerava očuvanje mase. Raspon granskih gubitaka provjerava energijsku kompatibilnost. Monotonost \(Q_B(D_B)\) dodatna je fizikalna provjera osjetljivosti.


In [ ]:
reynolds_branches = np.array([4*q/(np.pi*b["D"]*nu) for q,b in zip(flows,branches)])
assert np.all(reynolds_branches>4000)
assert abs(flows.sum()-Q_total) < 1e-10
assert np.ptp(losses) < 1e-10
assert abs(history[-1,2]) < 1e-10
assert np.all(np.diff(flow_sensitivity[:,1]) > 0)
assert np.allclose(flow_sensitivity.sum(axis=1), Q_total, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="vanjska iteracija", ylabel=r"$|\sum Q_i-Q|$ (m³/s)", title="Rezidual nelinearne mreže")
for i, b in enumerate(branches):
    axes[1].plot(1e3*D2_values, 1e3*flow_sensitivity[:,i], label=f"$Q_{b['name']}$")
axes[1].set(xlabel="promjer grane B (mm)", ylabel="granski protok (L/s)", title="Preraspodjela cijele mreže")
axes[1].legend()
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()

## 4. Z3: uravnoteženje dviju paralelnih grana

Ovo je **zaseban dvogranski model iz zadatka**, s već zadanim konstantnim otporima, a ne prethodna trogranska geometrija. Grane spajaju zajedničke čvorove A i B. $R_1=12000$ s²/m⁵, $R_2=48000$ s²/m⁵ i $Q=0{,}020$ m³/s. Regulacija održava ukupni protok i osigurava potrebnu visinu. Cilj je jednak protok u oba ogranka dodavanjem pasivnog ventila u samo jednu granu; ventil dodaje gubitak $h_{g,V}=R_vQ_i^2$. Predvidi gdje ventil treba biti prije računa.

In [ ]:
R3 = np.array([12000.,48000.])
Q3 = 0.020
q3 = np.full(2,Q3/2)
valve_branch3 = int(np.argmin(R3))
Rv3 = np.max(R3)-np.min(R3)
effective_R3 = R3.copy();effective_R3[valve_branch3] += Rv3
losses3 = effective_R3*q3**2
valve_loss3 = Rv3*q3[valve_branch3]**2
print(f"Ventil u grani {valve_branch3+1}; Rv = {Rv3:.0f} s²/m⁵")
print(f"Protoci = {1000*q3} L/s; zajednički gubitci = {losses3} m; ventil = {valve_loss3:.2f} m")
assert valve_branch3==0 and Rv3>=0
assert np.isclose(q3.sum(),0.020,atol=1e-14)
assert np.allclose(losses3,[4.8,4.8],atol=1e-12)
assert np.isclose(valve_loss3,3.6,atol=1e-12)
assert R3[0]-R3[1]<0  # Sam ventil u grani 2 tražio bi negativan dodatni otpor.

## 5. Z6: noćno punjenje vodospreme

Ovo je **zaseban model iz zadatka**, bez paralelnih grana iz prethodnih odjeljaka. Naselje stalno troši $Q_n=12$ L/s. Danas ga crpka uz jezero opskrbljuje izravno vodom $L_A=3{,}0$ km ($λ=0{,}019$), uz pretlak $3{,}0$ bar u točki N na $z_N=20$ m. Prema planu crpka bi noću, 8 sati, vodom $L_1=1{,}2$ km ($λ=0{,}019$) punila vodospremu s razinom $z_V=62$ m. Naselje bi tada dobivalo vodu gravitacijski vodom $L_3=2{,}0$ km ($λ=0{,}020$). Svi vodovi imaju $D=0{,}200$ m; lokalni gubitci su zanemareni. Ukupna učinkovitost agregata je 0,65, a nastavne cijene struje su 0,20 €/kWh danju (16 h) i 0,12 €/kWh noću (8 h).

Predvidi prije računa: troši li noćno punjenje više ili manje energije od izravne opskrbe? Pada li zato i trošak?

In [ ]:
def head_loss6(Q, L, lam, D=0.200):
    v = 4*Q/(np.pi*D**2)
    return lam*L/D*v**2/(2*g), v**2/(2*g)

Qn6, zN6, pM6, zV6, eta6 = 0.012, 20.0, 3.0e5, 62.0, 0.65
c_day6, c_night6 = 0.20, 0.12
hA6, vhA6 = head_loss6(Qn6, 3000.0, 0.019)
HA6 = zN6 + pM6/(1000*g) + vhA6 + hA6
PA6 = 1000*g*Qn6*HA6/eta6                     # W, 24 h dnevno
EA6 = PA6*24/1000                             # kWh/dan
CA6 = PA6/1000*(16*c_day6 + 8*c_night6)       # €/dan
QB6 = Qn6*24/8                                # cijela dnevna potrošnja u 8 h
h1_6, _ = head_loss6(QB6, 1200.0, 0.019)
HB6 = zV6 + h1_6
PB6 = 1000*g*QB6*HB6/eta6
EB6 = PB6*8/1000
CB6 = EB6*c_night6
c_star6 = PA6/1000*16*c_day6/(EB6 - PA6/1000*8)
h3_6, vh3_6 = head_loss6(Qn6, 2000.0, 0.020)
def p_N6(level):                              # pretlak u N pri gravitacijskoj opskrbi, bar
    return (level - zN6 - vh3_6 - h3_6)*1000*g/1e5
V_min6 = Qn6*16*3600
print(f"izravno: Hp = {HA6:.3f} m; Pel = {PA6/1000:.3f} kW; E = {EA6:.1f} kWh; trošak = {CA6:.2f} €/dan")
print(f"noćno:   Q = {QB6*1000:.1f} L/s; Hp = {HB6:.3f} m; Pel = {PB6/1000:.3f} kW; E = {EB6:.1f} kWh; trošak = {CB6:.2f} €/dan")
print(f"granična noćna cijena = {c_star6:.4f} €/kWh; pN = {p_N6(zV6):.3f} bar; V_min = {V_min6:.0f} m³")
assert abs(HA6-52.7)<0.05 and abs(PA6/1000-9.55)<0.005 and abs(CA6-39.7)<0.05
assert abs(HB6-69.6)<0.05 and abs(PB6/1000-37.8)<0.05 and abs(CB6-36.3)<0.05
assert abs(c_star6-0.135)<0.0005 and abs(p_N6(zV6)-3.97)<0.005 and abs(V_min6-691)<0.5
# Neovisne provjere: pri graničnoj cijeni troškovi su jednaki; noćni višak puni dnevnu potrošnju.
assert np.isclose(PA6/1000*(16*c_day6+8*c_star6), EB6*c_star6, rtol=1e-12)
assert np.isclose((QB6-Qn6)*8*3600, V_min6, rtol=1e-12)
assert EB6 > EA6 and CB6 < CA6
# Osjetljivost: koliko smije pasti razina u vodospremi prije nego što pretlak padne ispod 3 bar?
drop_allowed6 = zV6 - (zN6 + pM6/(1000*g) + vh3_6 + h3_6)
print(f"razina smije pasti najviše {drop_allowed6:.2f} m")
assert np.isclose(p_N6(zV6-drop_allowed6), 3.0, atol=1e-12)
prices6 = np.linspace(0.05, 0.20, 61)
fig,axes=plt.subplots(1,2,figsize=(10,3.8))
axes[0].plot(prices6, PA6/1000*(16*c_day6+8*prices6), label="izravna opskrba", color="#1565c0")
axes[0].plot(prices6, EB6*prices6, "--", label="noćno punjenje", color="#1e8449")
axes[0].axvline(c_star6, color="#3a4a56", lw=0.8)
axes[0].set(xlabel="noćna cijena (€/kWh)", ylabel="dnevni trošak (€)", title="Z6: isplativost prema noćnoj cijeni")
axes[0].legend(fontsize=8)
levels6 = np.linspace(zV6-12, zV6, 50)
axes[1].plot(zV6-levels6, [p_N6(z) for z in levels6], color="#c0392b")
axes[1].axhline(3.0, color="#3a4a56", lw=0.8, ls=":")
axes[1].set(xlabel="pad razine u vodospremi (m)", ylabel="pretlak u N (bar)", title="Gravitacijska opskrba")
for ax in axes: ax.grid(ls=":",alpha=.4)
plt.tight_layout();plt.show()

## 6. Protumači

1. Zašto povećanje promjera grane B u trogranskom pokusu mijenja i protoke ostalih grana? Što bi se izgubilo kad bi se promijenila samo brzina u grani B?
2. Promijeni lokalni koeficijent grane B za ±10 %. Kako se mijenja podjela protoka? Zašto taj zadani raspon nije automatski standardna mjerna nesigurnost?
3. Zašto ventil iz Z3 pripada grani manjeg otpora? Što bi se dogodilo s ukupnim protokom kada crpka ne bi mogla osigurati veću potrebnu visinu?
4. Noćno punjenje troši oko trećinu više energije, a ipak je jeftinije. Koja dva učinka određuju taj ishod i koji od njih ovisi o tarifi?
5. Koliko smije pasti razina u vodospremi prije nego što pretlak u N padne ispod 3 bar? Kako bi na tu granicu utjecala vodosprema veće tlocrtne površine?
6. Koja se od pretpostavki stalne učinkovitosti, konstantnih otpora ili turbulentnog režima mora ponovno provjeriti pri većoj promjeni radne točke?